# LoRA ViT — Train & Compare
Trains multiple LoRA configs on Food-101, saves checkpoints, then compares them side by side.

**Before running:** Runtime → Change runtime type → T4 GPU

## 1. Install deps (then Runtime → Restart runtime)

In [ ]:
!pip install -q --force-reinstall \
    'jsonargparse[signatures]==4.23.1' \
    'peft>=0.4.0' \
    'transformers>=4.30.0' \
    'torchmetrics>=1.0.0'
print('Done — now Restart runtime, then continue from cell 3.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Clone repos & upload scripts

In [1]:
import os

if not os.path.exists('/content/peft-vit'):
    !git clone https://github.com/rezaakb/peft-vit /content/peft-vit

if not os.path.exists('/content/lora-vit-food101'):
    !git clone https://github.com/Mtlukasik/lora-vit-food101 /content/lora-vit-food101

# Copy train_lora.py from your repo to /content (or upload manually)
!cp /content/lora-vit-food101/train_lora.py /content/train_lora.py

print('Ready.')
!ls /content/

Cloning into '/content/peft-vit'...
remote: Enumerating objects: 427, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 427 (delta 80), reused 75 (delta 75), pack-reused 327 (from 1)
Receiving objects: 100% (427/427), 2.80 MiB | 31.48 MiB/s, done.
Resolving deltas: 100% (281/281), done.
Cloning into '/content/lora-vit-food101'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 13 (delta 3), reused 13 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 10.65 KiB | 10.65 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Ready.
lora-vit-food101  peft-vit  sample_data  train_lora.py


## 3. Config — edit your runs here

In [2]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/vit_lora_food101'

def get_save_dir(run_cfg, stage='finetuned', dataset='food101'):
    model_name = 'vitb16-in21k'  # short name for the base model
    mods = '-'.join(run_cfg['target_modules'])
    run_id = f"r{run_cfg['rank']}_ep{run_cfg['epochs']}_{mods}"
    if stage == 'pretrained':
        folder = f"pretrained_{model_name}"
    else:
        folder = f"finetuned_{model_name}_{dataset}"
    return f"{BASE_DIR}/deterministic/{folder}/{run_id}"

RUNS = [
    {'rank': 4,  'epochs': 20, 'lr': 0.01, 'batch_size': 64, 'target_modules': ['query', 'value']},
    {'rank': 8,  'epochs': 20, 'lr': 0.01, 'batch_size': 64, 'target_modules': ['query', 'value']},
    {'rank': 16, 'epochs': 20, 'lr': 0.01, 'batch_size': 64, 'target_modules': ['query', 'value', 'key']},
]

# Attach a save_dir to each run
for run in RUNS:
    run['save_dir'] = get_save_dir(run, stage='finetuned', dataset='food101')
    print(run['save_dir'])


Mounted at /content/drive
/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r4_ep20_query-value
/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r8_ep20_query-value
/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r16_ep20_query-value-key


In [4]:
import glob
def load_finetuned(meta):
    run_dir = os.path.dirname(meta['_path'])
    base = ViTForImageClassification.from_pretrained(
        meta['model_name'], num_labels=meta['num_classes'], ignore_mismatched_sizes=True
    )
    cfg = LoraConfig(
        r=meta['rank'], lora_alpha=meta['lora_alpha'],
        target_modules=meta['target_modules'], lora_dropout=0.1, bias='none'
    )
    model = get_peft_model(base, cfg)
    model.load_state_dict(torch.load(os.path.join(run_dir, 'best.pt'), map_location=DEVICE))
    return model.to(DEVICE)
models = {} # name-> model
meta_files = sorted(glob.glob(f'{BASE_DIR}/**/meta.json', recursive=True))


In [5]:
meta_files

['/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r4_ep20_query-value/lora_r4_alpha8_ep20_query_value/meta.json',
 '/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r8_ep20_query-value/lora_r8_alpha16_ep20_query_value/meta.json']

In [9]:
import json, glob, os
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('mateusz_token'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Transforms ────────────────────────────────────────────────────────────────
val_tf = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ── Food-101 test set ─────────────────────────────────────────────────────────
food_ds = torchvision.datasets.Food101('/content/data', split='test', transform=val_tf, download=True)
food_dl = DataLoader(food_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f'Food-101 test: {len(food_ds)} images')

IMAGENET_DIR = '/content/drive/MyDrive/vit_lora_food101/imagenette_val'

if not os.path.exists(IMAGENET_DIR):
    print('Downloading Imagenette (~100MB)...')
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz -P /content/
    !tar -xzf /content/imagenette2-320.tgz -C /content/
    import shutil
    shutil.copytree('/content/imagenette2-320/val', IMAGENET_DIR)
    print(f'Saved to {IMAGENET_DIR}')
else:
    print(f'Imagenette val found at {IMAGENET_DIR}')

imagenet_ds = torchvision.datasets.ImageFolder(IMAGENET_DIR, transform=val_tf)
imagenet_dl = DataLoader(imagenet_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f'Imagenette val: {len(imagenet_ds)} images, {len(imagenet_ds.classes)} classes')


100%|██████████| 5.00G/5.00G [02:40<00:00, 31.1MB/s]


Food-101 test: 25250 images
Imagenette val found at /content/drive/MyDrive/vit_lora_food101/imagenette_val
Imagenette val: 3925 images, 10 classes

────────────────────────────────────────────────────────────
Run: lora_r4_alpha8_ep20_query_value


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

...

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



────────────────────────────────────────────────────────────
Run: lora_r8_alpha16_ep20_query_value


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r8_ep20_query-value/lora_r8_alpha16_ep20_query_value/best.pt'

In [11]:

import json
if not meta_files:
    print('No trained runs found. Train models first.')
else:
    results = []
    for path in meta_files:

          try:
            with open(path) as f:
              meta = json.load(f)
              meta['_path'] = path
              print(f"\n{'─'*60}")
              print(f"Run: {meta['run_name']}")

              models[path.split('/')[-2]] = load_finetuned(meta)
          except:
            continue


────────────────────────────────────────────────────────────
Run: lora_r4_alpha8_ep20_query_value


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



────────────────────────────────────────────────────────────
Run: lora_r8_alpha16_ep20_query_value


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
m = models['lora_r4_alpha8_ep20_query_value']


In [16]:
for name, param in m.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

base_model.model.vit.encoder.layer.0.attention.attention.query.lora_A.default.weight torch.Size([4, 768])
base_model.model.vit.encoder.layer.0.attention.attention.query.lora_B.default.weight torch.Size([768, 4])
base_model.model.vit.encoder.layer.0.attention.attention.value.lora_A.default.weight torch.Size([4, 768])
base_model.model.vit.encoder.layer.0.attention.attention.value.lora_B.default.weight torch.Size([768, 4])
base_model.model.vit.encoder.layer.1.attention.attention.query.lora_A.default.weight torch.Size([4, 768])
base_model.model.vit.encoder.layer.1.attention.attention.query.lora_B.default.weight torch.Size([768, 4])
base_model.model.vit.encoder.layer.1.attention.attention.value.lora_A.default.weight torch.Size([4, 768])
base_model.model.vit.encoder.layer.1.attention.attention.value.lora_B.default.weight torch.Size([768, 4])
base_model.model.vit.encoder.layer.2.attention.attention.query.lora_A.default.weight torch.Size([4, 768])
base_model.model.vit.encoder.layer.2.attention

## 4. Train all runs

In [ ]:
import subprocess, sys, json, os
from datetime import datetime

def build_cmd(run_cfg, save_dir):
    cmd = [
        sys.executable, '/content/train_lora.py',
        '--rank',       str(run_cfg['rank']),
        '--epochs',     str(run_cfg['epochs']),
        '--lr',         str(run_cfg['lr']),
        '--batch_size', str(run_cfg['batch_size']),
        '--save_dir',   save_dir,
        '--target_modules', *run_cfg['target_modules'],
    ]
    if 'lora_alpha' in run_cfg:
        cmd += ['--lora_alpha', str(run_cfg['lora_alpha'])]
    return cmd

env = os.environ.copy()
env['PYTHONPATH'] = '/content/peft-vit/src:/content/peft-vit'
import os
from datetime import datetime

for i, run_cfg in enumerate(RUNS):
    mods = ' '.join(run_cfg['target_modules'])
    save_dir = run_cfg['save_dir']
    r = run_cfg['rank']
    ep = run_cfg['epochs']
    lr = run_cfg['lr']
    bs = run_cfg['batch_size']

    print(f'\n{"="*60}')
    print(f'RUN {i+1}/{len(RUNS)} | r={r} epochs={ep} modules={mods}')
    print(f'{"="*60}')

    !PYTHONPATH=/content/peft-vit/src:/content/peft-vit \
        python /content/train_lora.py \
        --rank {r} --epochs {ep} --lr {lr} --batch_size {bs} \
        --save_dir {save_dir} --target_modules {mods}

print('\nAll runs complete.')


RUN 1/3 | r=4 epochs=20 modules=query value

Run: lora_r4_alpha8_ep20_query_value
Device: cuda | Rank: 4 | Alpha: 8 | Epochs: 20
LR: 0.01 | Batch: 64 | Modules: ['query', 'value']
Save dir: /content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r4_ep20_query-value/lora_r4_alpha8_ep20_query_value

Train: 75750 | Val: 25250

Loading weights: 100% 198/198 [00:00<00:00, 881.71it/s]
ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
trainable params: 147,456 || all params: 86,02

## 5. Compare results

In [11]:
import json, glob, os
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('mateusz_token'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Transforms ────────────────────────────────────────────────────────────────
val_tf = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ── Food-101 test set ─────────────────────────────────────────────────────────
food_ds = torchvision.datasets.Food101('/content/data', split='test', transform=val_tf, download=True)
food_dl = DataLoader(food_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f'Food-101 test: {len(food_ds)} images')

IMAGENET_DIR = '/content/drive/MyDrive/vit_lora_food101/imagenette_val'

if not os.path.exists(IMAGENET_DIR):
    print('Downloading Imagenette (~100MB)...')
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz -P /content/
    !tar -xzf /content/imagenette2-320.tgz -C /content/
    import shutil
    shutil.copytree('/content/imagenette2-320/val', IMAGENET_DIR)
    print(f'Saved to {IMAGENET_DIR}')
else:
    print(f'Imagenette val found at {IMAGENET_DIR}')

imagenet_ds = torchvision.datasets.ImageFolder(IMAGENET_DIR, transform=val_tf)
imagenet_dl = DataLoader(imagenet_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f'Imagenette val: {len(imagenet_ds)} images, {len(imagenet_ds.classes)} classes')

# ── Eval helper ───────────────────────────────────────────────────────────────
def evaluate(model, dl, label=''):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for imgs, labels in dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(imgs).logits
            correct += (logits.argmax(1) == labels).sum().item()
            n += len(labels)
    acc = 100 * correct / n
    print(f'  {label:<45} {acc:.2f}%')
    return acc

# ── Load fine-tuned model (Food-101 head) ─────────────────────────────────────
def load_finetuned(meta):
    run_dir = os.path.dirname(meta['_path'])
    base = ViTForImageClassification.from_pretrained(
        meta['model_name'], num_labels=meta['num_classes'], ignore_mismatched_sizes=True
    )
    cfg = LoraConfig(
        r=meta['rank'], lora_alpha=meta['lora_alpha'],
        target_modules=meta['target_modules'], lora_dropout=0.1, bias='none'
    )
    model = get_peft_model(base, cfg)
    model.load_state_dict(torch.load(os.path.join(run_dir, 'best.pt'), map_location=DEVICE))
    return model.to(DEVICE)

# ── Load same LoRA backbone but swap head to ImageNet-1k (1000 classes) ───────
# This measures how much the backbone representations drifted after fine-tuning
def load_imagenet_head(meta):
    run_dir = os.path.dirname(meta['_path'])
    base = ViTForImageClassification.from_pretrained(
        meta['model_name'], num_labels=meta['num_classes'], ignore_mismatched_sizes=True
    )
    cfg = LoraConfig(
        r=meta['rank'], lora_alpha=meta['lora_alpha'],
        target_modules=meta['target_modules'], lora_dropout=0.1, bias='none'
    )
    model = get_peft_model(base, cfg)
    model.load_state_dict(torch.load(os.path.join(run_dir, 'best.pt'), map_location=DEVICE))
    # Swap classifier to pretrained IN-1k head
    pretrained_in1k = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
    model.base_model.model.classifier = pretrained_in1k.classifier
    return model.to(DEVICE)

# ── Run comparison ────────────────────────────────────────────────────────────
meta_files = sorted(glob.glob(f'{BASE_DIR}/**/meta.json', recursive=True))
if not meta_files:
    print('No trained runs found. Train models first.')
else:
    results = []
    for path in meta_files:
        if '8' not in path.split():
          with open(path) as f:
              meta = json.load(f)
          meta['_path'] = path
          print(f"\n{'─'*60}")
          print(f"Run: {meta['run_name']}")

          model = load_finetuned(meta)
          food_acc = evaluate(model, food_dl, 'Food-101 test (finetuned head)')
          del model; torch.cuda.empty_cache()

          model = load_imagenet_head(meta)
          in1k_acc = evaluate(model, imagenet_dl, 'ImageNet-1k val (pretrained head)')
          del model; torch.cuda.empty_cache()

          results.append({
              'run':     meta['run_name'],
              'rank':    meta['rank'],
              'modules': '+'.join(meta['target_modules']),
              'epochs':  meta['epochs'],
              'food101': food_acc,
              'in1k':    in1k_acc,
              'mean':    (food_acc + in1k_acc) / 2,
          })

    results.sort(key=lambda x: x['mean'], reverse=True)
    print(f"\n{'='*72}")
    print(f"{'RUN':<35} {'r':>4} {'Food-101':>10} {'IN-1k':>10} {'MEAN':>8}")
    print(f"{'─'*72}")
    for r in results:
        print(f"{r['run']:<35} {r['rank']:>4} {r['food101']:>9.2f}% {r['in1k']:>9.2f}% {r['mean']:>7.2f}%")
    print(f"{'='*72}")
    print('\nPaper reference (trained on CIFAR-100, not directly comparable):')
    print('  r=4  : CIFAR-100=87.91%  IN-1k=66.82%  MEAN=77.37%')
    print('  r=8  : CIFAR-100=88.27%  IN-1k=65.99%  MEAN=77.13%')
    print('  r=16 : CIFAR-100=87.84%  IN-1k=65.06%  MEAN=76.45%')


Food-101 test: 25250 images
Imagenette val found at /content/drive/MyDrive/vit_lora_food101/imagenette_val
Imagenette val: 3925 images, 10 classes

────────────────────────────────────────────────────────────
Run: lora_r4_alpha8_ep20_query_value


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_172/3348229858.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Food-101 test (finetuned head)                46.70%


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  ImageNet-1k val (pretrained head)             9.10%

────────────────────────────────────────────────────────────
Run: lora_r8_alpha16_ep20_query_value


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/vit_lora_food101/deterministic/finetuned_vitb16-in21k_food101/r8_ep20_query-value/lora_r8_alpha16_ep20_query_value/best.pt'

## 6. Plot training curves

In [ ]:
import json, glob
import matplotlib.pyplot as plt

meta_files = sorted(glob.glob(f'{SAVE_DIR}/**/meta.json', recursive=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for path in meta_files:
    with open(path) as f:
        meta = json.load(f)
    label = f'r={meta["rank"]} {"+".join(meta["target_modules"])}'
    epochs = range(1, len(meta['history']['train_loss']) + 1)
    ax1.plot(epochs, meta['history']['train_loss'], marker='o', markersize=3, label=label)
    ax2.plot(epochs, meta['history']['val_acc'],   marker='o', markersize=3, label=label)

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.set_title('Val Accuracy (Food-101)'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Acc %'); ax2.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'Saved to {SAVE_DIR}/training_curves.png')

## 7. Save checkpoints to Google Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/checkpoints /content/drive/MyDrive/lora-vit-food101-checkpoints